In [1]:
import os 
os.chdir("C:\\Users\\lenovo\\Desktop\\sb miniproject")

In [12]:
#!/usr/bin/env python3
"""
CRISPR/Cas9 DSB repair kinetics: executable starter simulation.

Fits three observable states to the supplied paper Source Data workbook:
    Intact DNA, DSB (Direct DSB + Processed DSB), and Indels.

IMPORTANT:
- This is a simplified phenomenological model, not the paper's exact model.
- The workbook's actual column names are checked before fitting.
- Replicate measurements are summarized at each time point for fitting.
- Requires Python 3.10+ and: numpy, pandas, scipy, matplotlib, openpyxl
- Run:
    python crispr_simulation.py --excel 41467_2024_49410_MOESM6_ESM.xlsx
"""
from __future__ import annotations
import argparse
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

TARGETS = ["Psy1", "CRTISO", "PhyB2"]
REQUIRED_COLUMNS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]


def load_workbook(excel_path:"C:\\Users\\lenovo\\Desktop\\sb miniproject"):
    """Load target-specific 72-hour sheets and infer time from sample names."""
    xls = pd.ExcelFile(excel_path)
    records = []
    for target in TARGETS:
        sheet = f"{target} 72Hr"
        if sheet not in xls.sheet_names:
            raise ValueError(
                f"Missing sheet '{sheet}'. Workbook sheets: {xls.sheet_names}"
            )
        df = pd.read_excel(excel_path, sheet_name=sheet)
        # Find sample/name column; Excel may label it 'Unnamed: 0'.
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0],
        )
        df = df.rename(columns={sample_col: "sample"})
        missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise ValueError(
                f"Sheet '{sheet}' is missing columns {missing}. "
                f"Available columns: {list(df.columns)}"
            )
        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce",
        )
        # Exclude no-sgRNA/Cas9-only controls from the targeted kinetic fit.
        control_mask = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~control_mask].copy()
        for col in REQUIRED_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(subset=REQUIRED_COLUMNS)
        df["total"] = df[REQUIRED_COLUMNS].sum(axis=1)
        df = df[df["total"] > 0]
        df["intact_frac"] = df["Intact"] / df["total"]
        df["dsb_frac"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["indels_frac"] = df["Indels"] / df["total"]
        df["target"] = target
        records.append(df[["target", "sample", "time_h", "Intact", "Indels",
                           "Direct DSB", "Processed DSB", "total",
                           "intact_frac", "dsb_frac", "indels_frac"]])
    raw = pd.concat(records, ignore_index=True)
    return raw


def rhs(_t, y, k_cut, k_indel, k_repair):
    """Three-state ODE: Intact <-> DSB -> Indels."""
    intact, dsb, indels = y
    return (
        -k_cut * intact + k_repair * dsb,
        k_cut * intact - (k_indel + k_repair) * dsb,
        k_indel * dsb,
    )


def simulate(times, rates):
    """Simulate fractions at requested times; state order Intact, DSB, Indels."""
    times = np.asarray(times, dtype=float)
    if len(times) == 0:
        return np.empty((0, 3))
    order = np.argsort(times)
    ts = times[order]

In [13]:
#!/usr/bin/env python3
"""
Eight-compartment CRISPR/Cas9 kinetic simulation.

Compartments:
  S   = intact / susceptible DNA
  E   = Cas9-bound / engaged DNA
  Id  = direct double-strand break (DSB)
  Ip  = processed DSB
  R   = precisely repaired DNA
  Ins = insertion outcomes
  Del = deletion outcomes
  MH  = microhomology-mediated deletion outcomes

This is a mechanistic hypothesis model. The source workbook directly measures
only broad observable classes, not all eight compartments independently.
Therefore, this script simulates the eight states but DOES NOT claim that all
eight states/parameters are identifiable from the paper's data.

Requirements:
  pip install numpy pandas scipy matplotlib openpyxl

Run:
  python crispr_8compartment_simulation.py --excel 41467_2024_49410_MOESM6_ESM.xlsx

Outputs:
  eight_state_simulation.png
  eight_state_trajectories.csv
  eight_state_parameters.csv
  eight_state_observed_vs_aggregated.csv

Optional fit:
  --fit-observed
Fit only aggregate observables (Intact, total DSB, total Indels) to the source
data, while the extra compartments are constrained by model assumptions.
"""
from __future__ import annotations
import argparse
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

TARGETS = ["Psy1", "CRTISO", "PhyB2"]
OBS_COLS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]


def load_target_data(excel_path:"C:\\Users\\lenovo\\Desktop\\sb miniproject"):
    xls = pd.ExcelFile(excel_path)
    all_data = []
    for target in TARGETS:
        sheet = f"{target} 72Hr"
        if sheet not in xls.sheet_names:
            raise ValueError(f"Missing sheet {sheet!r}. Found: {xls.sheet_names}")
        df = pd.read_excel(excel_path, sheet_name=sheet)
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0],
        )
        df = df.rename(columns={sample_col: "sample"})
        missing = [c for c in OBS_COLS if c not in df.columns]
        if missing:
            raise ValueError(f"{sheet}: missing {missing}; columns={list(df.columns)}")
        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce",
        )
        controls = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~controls].copy()
        for c in OBS_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=OBS_COLS)
        df["total"] = df[OBS_COLS].sum(axis=1)
        df = df[df["total"] > 0]
        df["intact_obs"] = df["Intact"] / df["total"]
        df["dsb_obs"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["indels_obs"] = df["Indels"] / df["total"]
        df["target"] = target
        all_data.append(df[["target", "sample", "time_h", "intact_obs",
                            "dsb_obs", "indels_obs"]])
    return pd.concat(all_data, ignore_index=True)


def rhs(_t, y, p):
    """Eight-state ODE; rates are per hour."""
    S, E, Id, Ip, R, Ins, Del, MH = y
    k_on, k_off, k_cut, k_proc, k_precise, k_ins, k_del, k_mh, k_rebind = (
        p[k] for k in ("k_on", "k_off", "k_cut", "k_proc", "k_precise",
                       "k_ins", "k_del", "k_mh", "k_rebind")
    )

In [14]:
args.outdir = Path(r"C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results")
args.outdir.mkdir(parents=True, exist_ok=True)

NameError: name 'args' is not defined

In [15]:
from pathlib import Path

OUTDIR = Path(r"C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results")
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Results will be saved at:", OUTDIR)

Results will be saved at: C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results


In [16]:
plt.savefig(OUTDIR / "eight_state_simulation.png", dpi=220)

<Figure size 640x480 with 0 Axes>

In [20]:
r"""
CRISPR/Cas9 eight-compartment simulation — ready-to-run version.

Put this script and the source-data Excel workbook in:
C:\Users\lenovo\Desktop\sb miniproject

Then run this entire file in Python/Jupyter/VS Code.
All output files are automatically saved in:
C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results

States:
S, E, Id, Ip, R, Ins, Del, MH

Scientific note:
The source workbook measures broad classes (Intact, Indels, Direct DSB,
Processed DSB), not all eight compartments independently. Default rates are
illustrative assumptions. The optional partial fit estimates only k_on,
k_cut, k_proc; other rates remain fixed assumptions.
"""

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

# -------------------- PATHS: already set for your computer --------------------
PROJECT_DIR = Path(r"C:\Users\lenovo\Desktop\sb miniproject")
EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
OUTDIR = PROJECT_DIR / "final_results"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Set True to fit three macro-rates to the measured broad classes.
# Leave False for the illustrative simulation using starting assumptions.
FIT_OBSERVED_DATA = False

TARGETS = ["Psy1", "CRTISO", "PhyB2"]
OBS_COLS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]


def load_target_data(excel_path):
    """Load target-specific 72-hour source-data sheets and convert counts to fractions."""
    if not excel_path.exists():
        raise FileNotFoundError(
            f"\nExcel workbook not found:\n{excel_path}\n\n"
            "Put 41467_2024_49410_MOESM6_ESM.xlsx inside your sb miniproject folder."
        )

    xls = pd.ExcelFile(excel_path)
    all_data = []

    for target in TARGETS:
        sheet = f"{target} 72Hr"
        if sheet not in xls.sheet_names:
            raise ValueError(f"Missing sheet '{sheet}'. Available sheets: {xls.sheet_names}")

        df = pd.read_excel(excel_path, sheet_name=sheet)
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0]
        )
        df = df.rename(columns={sample_col: "sample"})

        missing = [c for c in OBS_COLS if c not in df.columns]
        if missing:
            raise ValueError(
                f"Sheet '{sheet}' is missing columns {missing}.\n"
                f"Columns found: {list(df.columns)}"
            )

        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce"
        )
        is_control = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~is_control].copy()

        for c in OBS_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=OBS_COLS)
        df["total"] = df[OBS_COLS].sum(axis=1)
        df = df[df["total"] > 0].copy()

        df["intact_obs"] = df["Intact"] / df["total"]
        df["dsb_obs"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["indels_obs"] = df["Indels"] / df["total"]
        df["target"] = target
        all_data.append(df[[
            "target", "sample", "time_h", "Intact", "Indels",
            "Direct DSB", "Processed DSB", "total",
            "intact_obs", "dsb_obs", "indels_obs"
        ]])

    return pd.concat(all_data, ignore_index=True)


def model_rhs(_t, y, p):
    """Eight-compartment ODE system; rate units are per hour."""
    S, E, Id, Ip, R, Ins, Del, MH = y

    kon = p["k_on"]
    koff = p["k_off"]
    kcut = p["k_cut"]
    kproc = p["k_proc"]
    kprecise = p["k_precise"]
    kins = p["k_ins"]
    kdel = p["k_del"]
    kmh = p["k_mh"]
    krebind = p["k_rebind"]

    # Intact DNA engages Cas9; engaged DNA can be cleaved.
    dS = -kon * S + koff * E + kprecise * R + krebind * R
    dE = kon * S - (koff + kcut) * E
    dId = kcut * E - (kproc + kprecise) * Id
    dIp = kproc * Id - (kprecise + kins + kdel + kmh) * Ip
    dR = kprecise * (Id + Ip) - krebind * R
    dIns = kins * Ip
    dDel = kdel * Ip
    dMH = kmh * Ip
    return [dS, dE, dId, dIp, dR, dIns, dDel, dMH]


def simulate(times, params):
    """Return state trajectories in order S, E, Id, Ip, R, Ins, Del, MH."""
    times = np.asarray(times, dtype=float)
    if np.any(times < 0):
        raise ValueError("Simulation times must be non-negative.")
    unique_times = np.unique(np.concatenate(([0.0], times)))
    solution = solve_ivp(
        lambda t, y: model_rhs(t, y, params),
        (0.0, float(unique_times[-1])),
        [1., 0., 0., 0., 0., 0., 0., 0.],
        t_eval=unique_times,
        rtol=1e-8,
        atol=1e-10
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    lookup = {float(t): solution.y[:, i] for i, t in enumerate(solution.t)}
    return np.vstack([lookup[float(t)] for t in times])


def aggregate_states(states):
    """
    Approximate mapping to broad measured categories.
    Intact approx. = S + E + R; DSB = Id + Ip; indels = Ins + Del + MH.
    This mapping is a modelling assumption, not a direct measurement of each state.
    """
    intact = states[:, 0] + states[:, 1] + states[:, 4]
    dsb = states[:, 2] + states[:, 3]
    indels = states[:, 5] + states[:, 6] + states[:, 7]
    result = np.column_stack([intact, dsb, indels])
    totals = result.sum(axis=1, keepdims=True)
    return result / np.maximum(totals, 1e-12)


def default_params(target):
    # Illustrative initial rates, not published parameter estimates.
    rates = {
        "Psy1": {
            "k_on": .08, "k_off": .12, "k_cut": .20, "k_proc": .09,
            "k_precise": .05, "k_ins": .02, "k_del": .04, "k_mh": .01,
            "k_rebind": .01
        },
        "CRTISO": {
            "k_on": .10, "k_off": .10, "k_cut": .25, "k_proc": .12,
            "k_precise": .04, "k_ins": .02, "k_del": .05, "k_mh": .01,
            "k_rebind": .01
        },
        "PhyB2": {
            "k_on": .15, "k_off": .08, "k_cut": .35, "k_proc": .15,
            "k_precise": .03, "k_ins": .02, "k_del": .05, "k_mh": .01,
            "k_rebind": .01
        }
    }
    return rates[target].copy()


def fit_three_macro_rates(data, target, starting_params):
    """Fit only k_on, k_cut, k_proc; all other rates remain fixed assumptions."""
    d = data[data["target"] == target].groupby("time_h", as_index=False).agg(
        intact=("intact_obs", "mean"),
        dsb=("dsb_obs", "mean"),
        indels=("indels_obs", "mean")
    ).sort_values("time_h")

    times = d["time_h"].to_numpy(float)
    observed = d[["intact", "dsb", "indels"]].to_numpy(float)
    keys = ["k_on", "k_cut", "k_proc"]

    def residual(log_rates):
        p = starting_params.copy()
        for key, value in zip(keys, np.exp(log_rates)):
            p[key] = float(value)
        return (aggregate_states(simulate(times, p)) - observed).ravel()

    result = least_squares(
        residual,
        np.log([starting_params[k] for k in keys]),
        bounds=(np.log([1e-5] * 3), np.log([2.0] * 3)),
        max_nfev=3000
    )
    fitted_params = starting_params.copy()
    for key, value in zip(keys, np.exp(result.x)):
        fitted_params[key] = float(value)
    return fitted_params, bool(result.success)


def main():
    print("Loading real source data...")
    raw = load_target_data(EXCEL_FILE)
    raw.to_csv(OUTDIR / "source_72h_samples.csv", index=False)

    time_grid = np.linspace(0, 72, 361)
    all_trajectories = []
    parameter_rows = []
    comparison_rows = []
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

    state_names = ["S", "E", "Id", "Ip", "R", "Ins", "Del", "MH"]

    for ax, target in zip(axes, TARGETS):
        params = default_params(target)
        fit_success = None

        observed = raw[raw["target"] == target].groupby(
            "time_h", as_index=False
        ).agg(
            intact=("intact_obs", "mean"),
            dsb=("dsb_obs", "mean"),
            indels=("indels_obs", "mean")
        ).sort_values("time_h")

        if FIT_OBSERVED_DATA:
            params, fit_success = fit_three_macro_rates(raw, target, params)

        states = simulate(time_grid, params)
        aggregate = aggregate_states(states)

        # Save all eight states at every simulation time.
        for i, state_name in enumerate(state_names):
            for t, val in zip(time_grid, states[:, i]):
                all_trajectories.append({
                    "target": target, "time_h": float(t),
                    "state": state_name, "fraction": float(val)
                })

        # Save observed and model aggregate values at observed time points.
        obs_times = observed["time_h"].to_numpy(float)
        pred_at_obs = aggregate_states(simulate(obs_times, params))
        for idx, t in enumerate(obs_times):
            comparison_rows.append({
                "target": target, "time_h": float(t),
                "observed_intact": float(observed.iloc[idx]["intact"]),
                "model_intact_approx": float(pred_at_obs[idx, 0]),
                "observed_dsb": float(observed.iloc[idx]["dsb"]),
                "model_dsb": float(pred_at_obs[idx, 1]),
                "observed_indels": float(observed.iloc[idx]["indels"]),
                "model_indels": float(pred_at_obs[idx, 2])
            })

        parameter_rows.append({
            "target": target, **params,
            "rates_fitted_to_observations": FIT_OBSERVED_DATA,
            "fit_optimizer_success": fit_success
        })

        specs = [
            ("intact", aggregate[:, 0], "Intact (approx.)", "#2b8a3e"),
            ("dsb", aggregate[:, 1], "Total DSB", "#1971c2"),
            ("indels", aggregate[:, 2], "Total indels", "#d9480f")
        ]
        for col, curve, label, color in specs:
            ax.scatter(observed["time_h"], observed[col], s=22, color=color, alpha=.8)
            ax.plot(time_grid, curve, color=color, lw=1.8, label=label)
        title_suffix = "partial fit" if FIT_OBSERVED_DATA else "illustrative rates"
        ax.set_title(f"{target} ({title_suffix})")
        ax.set_xlabel("Time (hours)")
        ax.grid(alpha=.25)

    axes[0].set_ylabel("Fraction of measured molecules")
    axes[0].legend(fontsize=8)
    fig.suptitle("Eight-compartment CRISPR/Cas9 simulation vs. broad source-data classes", y=1.02)
    fig.tight_layout()
    fig.savefig(OUTDIR / "eight_state_simulation.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    pd.DataFrame(all_trajectories).to_csv(
        OUTDIR / "eight_state_trajectories.csv", index=False
    )
    pd.DataFrame(parameter_rows).to_csv(
        OUTDIR / "eight_state_parameters.csv", index=False
    )
    pd.DataFrame(comparison_rows).to_csv(
        OUTDIR / "observed_vs_model_aggregates.csv", index=False
    )

    # Also save a readable summary text file.
    summary = [
        "CRISPR/Cas9 eight-compartment simulation results",
        f"Excel source: {EXCEL_FILE}",
        f"Output folder: {OUTDIR}",
        f"Rates fitted to broad observations: {FIT_OBSERVED_DATA}",
        "",
        "States: S, E, Id, Ip, R, Ins, Del, MH",
        "Caution: the source data do not independently measure all eight states.",
        "Default rates are assumptions; if partial fitting is enabled, only three",
        "macro rates are fitted and the remaining rates stay fixed."
    ]
    (OUTDIR / "README_results.txt").write_text("\n".join(summary), encoding="utf-8")

    print("\nDONE! ALL RESULTS SAVED HERE:")
    print(OUTDIR)
    print("\nFiles created:")
    for file in sorted(OUTDIR.iterdir()):
        print(" -", file.name)


if __name__ == "__main__":
    main()


Loading real source data...

DONE! ALL RESULTS SAVED HERE:
C:\Users\lenovo\Desktop\sb miniproject\final_results

Files created:
 - eight_state_parameters.csv
 - eight_state_simulation.png
 - eight_state_trajectories.csv
 - observed_vs_model_aggregates.csv
 - README_results.txt
 - source_72h_samples.csv


In [21]:
r"""
Eight-compartment CRISPR/Cas9 simulation with a familiar eight-curve plot.

Place this script and the source Excel workbook in:
C:\Users\lenovo\Desktop\sb miniproject

Run:
  python crispr_eight_curves_real_data.py

All outputs are saved automatically to:
C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results

IMPORTANT:
The source workbook measures Intact, Indels, Direct DSB, and Processed DSB.
It does NOT independently measure all eight compartments. This script fits
three aggregate rates (k_on, k_cut, k_proc) to broad data classes while holding
other rates fixed at stated illustrative assumptions. The eight lines are model
predictions; experimental observations are shown only for measured categories.
"""
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

PROJECT_DIR = Path(r"C:\Users\lenovo\Desktop\sb miniproject")
EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
OUTDIR = PROJECT_DIR / "final_results"
# On non-Windows systems, use the script's folder so the script is testable there too.
if os.name != "nt":
    PROJECT_DIR = Path(__file__).resolve().parent
    EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
    OUTDIR = PROJECT_DIR / "final_results"
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGETS = ["Psy1", "CRTISO", "PhyB2"]
OBS_COLS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]
STATE_NAMES = [
    "Intact DNA (S)", "Cas9-engaged DNA (E)", "Direct DSB (Id)",
    "Processed DSB (Ip)", "Precisely repaired (R)", "Insertions (Ins)",
    "Deletions (Del)", "Microhomology deletions (MH)"
]


def load_data():
    if not EXCEL_FILE.exists():
        raise FileNotFoundError(
            f"Workbook not found: {EXCEL_FILE}\n"
            "Put the downloaded source-data Excel workbook in the project folder."
        )
    xls = pd.ExcelFile(EXCEL_FILE)
    chunks = []
    for target in TARGETS:
        sheet = f"{target} 72Hr"
        df = pd.read_excel(EXCEL_FILE, sheet_name=sheet)
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0]
        )
        df = df.rename(columns={sample_col: "sample"})
        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce"
        )
        controls = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~controls].copy()
        for c in OBS_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=OBS_COLS)
        df["total"] = df[OBS_COLS].sum(axis=1)
        df = df[df["total"] > 0].copy()
        df["obs_intact"] = df["Intact"] / df["total"]
        df["obs_dsb"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["obs_indels"] = df["Indels"] / df["total"]
        df["target"] = target
        chunks.append(df[[
            "target", "sample", "time_h", "Intact", "Indels",
            "Direct DSB", "Processed DSB", "total",
            "obs_intact", "obs_dsb", "obs_indels"
        ]])
    return pd.concat(chunks, ignore_index=True)


def rhs(t, y, p):
    S, E, Id, Ip, R, Ins, Del, MH = y
    kon, koff, kcut, kproc, kprec, kins, kdel, kmh, krebind = (
        p[k] for k in ["k_on", "k_off", "k_cut", "k_proc", "k_precise",
                       "k_ins", "k_del", "k_mh", "k_rebind"]
    )
    dS = -kon*S + koff*E + kprec*R + krebind*R
    dE = kon*S - (koff+kcut)*E
    dId = kcut*E - (kproc+kprec)*Id
    dIp = kproc*Id - (kprec+kins+kdel+kmh)*Ip
    dR = kprec*(Id+Ip) - krebind*R
    return [dS, dE, dId, dIp, dR, kins*Ip, kdel*Ip, kmh*Ip]


def simulate(times, p):
    times = np.asarray(times, dtype=float)
    unique = np.unique(np.concatenate(([0.0], times)))
    sol = solve_ivp(
        lambda t, y: rhs(t, y, p), (0.0, float(unique[-1])),
        [1., 0., 0., 0., 0., 0., 0., 0.], t_eval=unique,
        rtol=1e-8, atol=1e-10
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    lookup = {float(t): sol.y[:, i] for i, t in enumerate(sol.t)}
    return np.vstack([lookup[float(t)] for t in times])


def aggregate(states):
    # Approximate mapping to assay categories; not a direct state measurement.
    x = np.column_stack([
        states[:, 0] + states[:, 1] + states[:, 4],
        states[:, 2] + states[:, 3],
        states[:, 5] + states[:, 6] + states[:, 7]
    ])
    return x / np.maximum(x.sum(axis=1, keepdims=True), 1e-12)


def initial_params(target):
    return {
        "Psy1": dict(k_on=.08, k_off=.12, k_cut=.20, k_proc=.09,
                     k_precise=.05, k_ins=.02, k_del=.04, k_mh=.01, k_rebind=.01),
        "CRTISO": dict(k_on=.10, k_off=.10, k_cut=.25, k_proc=.12,
                       k_precise=.04, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01),
        "PhyB2": dict(k_on=.15, k_off=.08, k_cut=.35, k_proc=.15,
                      k_precise=.03, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01)
    }[target].copy()


def fit_macro_rates(target, obs):
    p = initial_params(target)
    t = obs["time_h"].to_numpy(float)
    yobs = obs[["intact", "dsb", "indels"]].to_numpy(float)
    fit_keys = ["k_on", "k_cut", "k_proc"]

    def residual(theta):
        pp = p.copy()
        for key, value in zip(fit_keys, np.exp(theta)):
            pp[key] = float(value)
        return (aggregate(simulate(t, pp)) - yobs).ravel()

    result = least_squares(
        residual, np.log([p[k] for k in fit_keys]),
        bounds=(np.log([1e-5]*3), np.log([2.0]*3)), max_nfev=4000
    )
    for key, value in zip(fit_keys, np.exp(result.x)):
        p[key] = float(value)
    return p, bool(result.success)


def main():
    raw = load_data()
    raw.to_csv(OUTDIR / "real_source_data_used.csv", index=False)
    summary = raw.groupby(["target", "time_h"], as_index=False).agg(
        intact=("obs_intact", "mean"),
        dsb=("obs_dsb", "mean"),
        indels=("obs_indels", "mean"),
        intact_sd=("obs_intact", "std"),
        dsb_sd=("obs_dsb", "std"),
        indels_sd=("obs_indels", "std"),
        n_replicates=("sample", "count")
    )
    summary.to_csv(OUTDIR / "real_data_summary.csv", index=False)

    grid = np.linspace(0, 72, 361)
    colors = ["#1971c2", "#fd7e14", "#2b8a3e", "#e03131",
              "#845ef7", "#795548", "#f06595", "#868e96"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.6), sharey=True)
    all_states, params_rows, comparisons = [], [], []

    for ax, target in zip(axes, TARGETS):
        obs = summary[summary["target"] == target].sort_values("time_h")
        params, success = fit_macro_rates(target, obs)
        states = simulate(grid, params)
        # Normalize for display as percent of model pool; these are model states.
        display = np.maximum(states, 0)
        display = display / np.maximum(display.sum(axis=1, keepdims=True), 1e-12) * 100

        for i, state_name in enumerate(STATE_NAMES):
            ax.plot(grid, display[:, i], color=colors[i], lw=2,
                    label=state_name if target == TARGETS[0] else None)
            for tt, vv in zip(grid, display[:, i]):
                all_states.append({"target": target, "time_h": tt,
                                   "state": state_name, "model_percent": vv})

        # Actual experimental dots for the broad measured classes only.
        # Use distinct marker styles and matching approximate state groups.
        obs_specs = [
            ("intact", 0, "Observed intact", "o"),
            ("dsb", 2, "Observed total DSB", "s"),
            ("indels", 5, "Observed indels", "^")
        ]
        pred_obs = aggregate(simulate(obs["time_h"].to_numpy(float), params))
        for col, pred_i, label, marker in obs_specs:
            ax.scatter(obs["time_h"], obs[col] * 100, s=25, marker=marker,
                       facecolors="none", edgecolors=colors[pred_i],
                       linewidths=1.1, label=label if target == TARGETS[0] else None,
                       zorder=4)
        rmse = float(np.sqrt(np.mean((
            pred_obs - obs[["intact", "dsb", "indels"]].to_numpy(float)
        )**2)))
        params_rows.append({
            "target": target, **params, "macro_fit_success": success,
            "aggregate_RMSE_fraction": rmse,
            "note": "Only k_on, k_cut, k_proc fitted; remaining rates are assumptions."
        })
        for i, tt in enumerate(obs["time_h"].to_numpy(float)):
            comparisons.append({
                "target": target, "time_h": tt,
                "observed_intact_fraction": obs.iloc[i]["intact"],
                "observed_dsb_fraction": obs.iloc[i]["dsb"],
                "observed_indels_fraction": obs.iloc[i]["indels"],
                "predicted_intact_approx": pred_obs[i, 0],
                "predicted_dsb": pred_obs[i, 1],
                "predicted_indels": pred_obs[i, 2]
            })
        ax.set_title(f"{target} (aggregate RMSE={rmse:.3f})")
        ax.set_xlabel("Time (hours)")
        ax.grid(alpha=.22)
    axes[0].set_ylabel("Fraction of model pool (%)")
    axes[0].legend(fontsize=7, ncol=2, loc="upper right")
    fig.suptitle(
        "Eight-compartment model predictions with real experimental observations",
        y=1.02
    )
    fig.tight_layout()
    fig.savefig(OUTDIR / "eight_curve_style_real_data.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    pd.DataFrame(all_states).to_csv(OUTDIR / "eight_state_trajectories.csv", index=False)
    pd.DataFrame(params_rows).to_csv(OUTDIR / "eight_state_parameters_and_fit.csv", index=False)
    pd.DataFrame(comparisons).to_csv(OUTDIR / "observed_vs_predicted_broad_classes.csv", index=False)
    (OUTDIR / "README_results.txt").write_text(
        "Eight-compartment model with real source-data observations.\n"
        "Curves are model predictions; open markers are experimental measurements\n"
        "for broad classes only. The source data do not independently measure all\n"
        "eight compartments. Only k_on, k_cut, k_proc are fitted; other rates\n"
        "remain illustrative assumptions. Do not describe all eight curves as\n"
        "experimentally measured or fully validated.\n", encoding="utf-8"
    )
    print("DONE. Results saved to:", OUTDIR)
    print("Files:")
    for f in sorted(OUTDIR.iterdir()):
        print(" -", f.name)
    print("\nFit summary:")
    print(pd.DataFrame(params_rows)[["target", "aggregate_RMSE_fraction", "macro_fit_success"]].to_string(index=False))


if __name__ == "__main__":
    main()


DONE. Results saved to: C:\Users\lenovo\Desktop\sb miniproject\final_results
Files:
 - eight_curve_style_real_data.png
 - eight_state_parameters.csv
 - eight_state_parameters_and_fit.csv
 - eight_state_simulation.png
 - eight_state_trajectories.csv
 - observed_vs_model_aggregates.csv
 - observed_vs_predicted_broad_classes.csv
 - README_results.txt
 - real_data_summary.csv
 - real_source_data_used.csv
 - source_72h_samples.csv

Fit summary:
target  aggregate_RMSE_fraction  macro_fit_success
  Psy1                 0.018046               True
CRTISO                 0.051580               True
 PhyB2                 0.049681               True


In [22]:
r"""
CRISPR/Cas9: separate plots for each of the eight compartments.

This script is for understanding the eight compartments one by one.
It uses the same eight-state ODE model and fits three aggregate rates to the
real source-data categories. It then creates:
  1) One separate PNG for each compartment (8 plots per target = 24 plots)
  2) One 8-panel figure per target (3 figures)
  3) CSVs of all simulated trajectories and fitted/assumed parameters

Put this script and the Excel workbook in:
C:\Users\lenovo\Desktop\sb miniproject

Run:
  python crispr_8_compartments_separate_plots.py

Results automatically save to:
C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results\separate_compartment_plots

Scientific caution:
The paper's source data measure broad categories, not all eight compartments
independently. Only k_on, k_cut and k_proc are fitted to aggregate observations;
other rates are illustrative assumptions. Separate compartment curves are model
predictions, not direct experimental measurements.
"""
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

PROJECT_DIR = Path(r"C:\Users\lenovo\Desktop\sb miniproject")
EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
OUTDIR = PROJECT_DIR / "final_results" / "separate_compartment_plots"

# Make this script runnable on non-Windows systems too.
if os.name != "nt":
    PROJECT_DIR = Path(__file__).resolve().parent
    EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
    OUTDIR = PROJECT_DIR / "final_results" / "separate_compartment_plots"

OUTDIR.mkdir(parents=True, exist_ok=True)
TARGETS = ["Psy1", "CRTISO", "PhyB2"]
OBS_COLS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]
STATES = [
    ("S", "S — Intact/susceptible DNA"),
    ("E", "E — Cas9-engaged DNA"),
    ("Id", "Id — Direct double-strand break"),
    ("Ip", "Ip — Processed double-strand break"),
    ("R", "R — Precisely repaired DNA"),
    ("Ins", "Ins — Insertions"),
    ("Del", "Del — Deletions"),
    ("MH", "MH — Microhomology-mediated deletions"),
]


def load_data():
    if not EXCEL_FILE.exists():
        raise FileNotFoundError(
            f"Excel workbook not found:\n{EXCEL_FILE}\n"
            "Put the workbook in the sb miniproject folder beside this script."
        )
    rows = []
    for target in TARGETS:
        df = pd.read_excel(EXCEL_FILE, sheet_name=f"{target} 72Hr")
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0]
        )
        df = df.rename(columns={sample_col: "sample"})
        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce"
        )
        controls = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~controls].copy()
        for c in OBS_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=OBS_COLS)
        df["total"] = df[OBS_COLS].sum(axis=1)
        df = df[df["total"] > 0].copy()
        df["intact"] = df["Intact"] / df["total"]
        df["dsb"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["indels"] = df["Indels"] / df["total"]
        df["target"] = target
        rows.append(df[["target", "sample", "time_h", "intact", "dsb", "indels"]])
    return pd.concat(rows, ignore_index=True)


def rhs(_t, y, p):
    S, E, Id, Ip, R, Ins, Del, MH = y
    kon, koff, kcut, kproc, kprec, kins, kdel, kmh, krebind = (
        p[k] for k in ["k_on", "k_off", "k_cut", "k_proc", "k_precise",
                       "k_ins", "k_del", "k_mh", "k_rebind"]
    )
    dS = -kon*S + koff*E + kprec*R + krebind*R
    dE = kon*S - (koff+kcut)*E
    dId = kcut*E - (kproc+kprec)*Id
    dIp = kproc*Id - (kprec+kins+kdel+kmh)*Ip
    dR = kprec*(Id+Ip) - krebind*R
    return [dS, dE, dId, dIp, dR, kins*Ip, kdel*Ip, kmh*Ip]


def simulate(times, p):
    times = np.asarray(times, dtype=float)
    unique = np.unique(np.concatenate(([0.0], times)))
    sol = solve_ivp(
        lambda t, y: rhs(t, y, p), (0.0, float(unique[-1])),
        [1., 0., 0., 0., 0., 0., 0., 0.], t_eval=unique,
        rtol=1e-8, atol=1e-10
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    lookup = {float(t): sol.y[:, i] for i, t in enumerate(sol.t)}
    return np.vstack([lookup[float(t)] for t in times])


def aggregate(states):
    # Approximate mapping to the broad classes measured in the source workbook.
    a = np.column_stack([
        states[:, 0] + states[:, 1] + states[:, 4],
        states[:, 2] + states[:, 3],
        states[:, 5] + states[:, 6] + states[:, 7]
    ])
    return a / np.maximum(a.sum(axis=1, keepdims=True), 1e-12)


def initial_params(target):
    table = {
        "Psy1": dict(k_on=.08, k_off=.12, k_cut=.20, k_proc=.09,
                     k_precise=.05, k_ins=.02, k_del=.04, k_mh=.01, k_rebind=.01),
        "CRTISO": dict(k_on=.10, k_off=.10, k_cut=.25, k_proc=.12,
                       k_precise=.04, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01),
        "PhyB2": dict(k_on=.15, k_off=.08, k_cut=.35, k_proc=.15,
                      k_precise=.03, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01)
    }
    return table[target].copy()


def fit_macro_rates(target, observed):
    p = initial_params(target)
    times = observed["time_h"].to_numpy(float)
    yobs = observed[["intact", "dsb", "indels"]].to_numpy(float)
    keys = ["k_on", "k_cut", "k_proc"]

    def residual(theta):
        pp = p.copy()
        for key, value in zip(keys, np.exp(theta)):
            pp[key] = float(value)
        return (aggregate(simulate(times, pp)) - yobs).ravel()

    result = least_squares(
        residual, np.log([p[k] for k in keys]),
        bounds=(np.log([1e-5]*3), np.log([2.0]*3)), max_nfev=4000
    )
    for key, value in zip(keys, np.exp(result.x)):
        p[key] = float(value)
    return p, bool(result.success)


def main():
    raw = load_data()
    raw.to_csv(OUTDIR / "experimental_data_used.csv", index=False)
    summary = raw.groupby(["target", "time_h"], as_index=False).agg(
        intact=("intact", "mean"), dsb=("dsb", "mean"),
        indels=("indels", "mean"), n_replicates=("sample", "count")
    )
    summary.to_csv(OUTDIR / "observed_summary.csv", index=False)

    grid = np.linspace(0, 72, 361)
    colors = ["#1971c2", "#fd7e14", "#2b8a3e", "#e03131",
              "#845ef7", "#795548", "#f06595", "#868e96"]
    trajectories, params_rows = [], []

    for target in TARGETS:
        obs = summary[summary["target"] == target].sort_values("time_h")
        params, success = fit_macro_rates(target, obs)
        states = np.maximum(simulate(grid, params), 0)
        states_pct = states / np.maximum(states.sum(axis=1, keepdims=True), 1e-12) * 100

        # Save all states to CSV.
        for j, (short, long_name) in enumerate(STATES):
            for t, value in zip(grid, states_pct[:, j]):
                trajectories.append({
                    "target": target, "time_h": t, "state": short,
                    "state_description": long_name, "model_percent": value
                })

            # One separate plot per compartment per target.
            fig, ax = plt.subplots(figsize=(8, 4.8))
            ax.plot(grid, states_pct[:, j], color=colors[j], lw=2.5)
            ax.set_title(f"{target}: {long_name}")
            ax.set_xlabel("Time (hours)")
            ax.set_ylabel("% of model pool (predicted)")
            ax.set_xlim(0, 72)
            ax.grid(alpha=.25)
            ax.text(
                .02, .96,
                "Model prediction — this compartment is not directly measured separately",
                transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=.3", facecolor="white", alpha=.8)
            )
            fig.tight_layout()
            fig.savefig(OUTDIR / f"{target}_{short}_separate.png", dpi=200, bbox_inches="tight")
            plt.close(fig)

        # One 8-panel plot per target for easy understanding.
        fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
        for j, ax in enumerate(axes.flat):
            ax.plot(grid, states_pct[:, j], color=colors[j], lw=2)
            ax.set_title(STATES[j][1], fontsize=9)
            ax.set_xlabel("Time (h)")
            ax.set_ylabel("% predicted")
            ax.grid(alpha=.2)
        fig.suptitle(f"{target}: eight compartments shown separately", y=1.02)
        fig.tight_layout()
        fig.savefig(OUTDIR / f"{target}_all_8_compartments_panel.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        params_rows.append({
            "target": target, **params, "macro_fit_success": success,
            "note": "Only k_on, k_cut, k_proc fitted; other rates are assumptions."
        })

    pd.DataFrame(trajectories).to_csv(OUTDIR / "all_8_state_trajectories.csv", index=False)
    pd.DataFrame(params_rows).to_csv(OUTDIR / "parameters_fitted_and_assumed.csv", index=False)
    (OUTDIR / "README.txt").write_text(
        "Separate eight-compartment plots for learning/interpretation.\\n"
        "Individual curves are model predictions, not direct experimental measurements.\\n"
        "Only k_on, k_cut, k_proc are fitted to aggregate observed classes; other rates are assumptions.\\n",
        encoding="utf-8"
    )
    print("DONE! All separate plots and CSV results saved to:")
    print(OUTDIR)
    print("Created 24 individual compartment plots + 3 eight-panel plots.")
    print("See README.txt for the scientific limitations.")


if __name__ == "__main__":
    main()


DONE! All separate plots and CSV results saved to:
C:\Users\lenovo\Desktop\sb miniproject\final_results\separate_compartment_plots
Created 24 individual compartment plots + 3 eight-panel plots.
See README.txt for the scientific limitations.


In [24]:
r"""
CRISPR/Cas9: separate plots for each of the eight compartments.

This script is for understanding the eight compartments one by one.
It uses the same eight-state ODE model and fits three aggregate rates to the
real source-data categories. It then creates:
  1) One separate PNG for each compartment (8 plots per target = 24 plots)
  2) One 8-panel figure per target (3 figures)
  3) CSVs of all simulated trajectories and fitted/assumed parameters

Put this script and the Excel workbook in:
C:\Users\lenovo\Desktop\sb miniproject

Run:
  python crispr_8_compartments_separate_plots.py

Results automatically save to:
C:\Users\lenovo\Desktop\sb miniproject\crispr_8state_results\separate_compartment_plots

Scientific caution:
The paper's source data measure broad categories, not all eight compartments
independently. Only k_on, k_cut and k_proc are fitted to aggregate observations;
other rates are illustrative assumptions. Separate compartment curves are model
predictions, not direct experimental measurements.
"""
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

PROJECT_DIR = Path(r"C:\Users\lenovo\Desktop\sb miniproject")
EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
OUTDIR = PROJECT_DIR / "final_results" / "separate_compartment_plots"

# Make this script runnable on non-Windows systems too.
if os.name != "nt":
    PROJECT_DIR = Path(__file__).resolve().parent
    EXCEL_FILE = PROJECT_DIR / "41467_2024_49410_MOESM6_ESM.xlsx"
    OUTDIR = PROJECT_DIR / "final_results" / "separate_compartment_plots"

OUTDIR.mkdir(parents=True, exist_ok=True)
TARGETS = ["Psy1", "CRTISO", "PhyB2"]
OBS_COLS = ["Intact", "Indels", "Direct DSB", "Processed DSB"]
STATES = [
    ("S", "S — Intact/susceptible DNA"),
    ("E", "E — Cas9-engaged DNA"),
    ("Id", "Id — Direct double-strand break"),
    ("Ip", "Ip — Processed double-strand break"),
    ("R", "R — Precisely repaired DNA"),
    ("Ins", "Ins — Insertions"),
    ("Del", "Del — Deletions"),
    ("MH", "MH — Microhomology-mediated deletions"),
]


def load_data():
    if not EXCEL_FILE.exists():
        raise FileNotFoundError(
            f"Excel workbook not found:\n{EXCEL_FILE}\n"
            "Put the workbook in the sb miniproject folder beside this script."
        )
    rows = []
    for target in TARGETS:
        df = pd.read_excel(EXCEL_FILE, sheet_name=f"{target} 72Hr")
        sample_col = next(
            (c for c in df.columns if str(c).strip().lower() in
             {"sample", "samples", "unnamed: 0", "sample name", "name"}),
            df.columns[0]
        )
        df = df.rename(columns={sample_col: "sample"})
        df["time_h"] = pd.to_numeric(
            df["sample"].astype(str).str.extract(r"_(\d+)h", expand=False),
            errors="coerce"
        )
        controls = df["sample"].astype(str).str.contains(
            r"control|cas9.?only|no.?sg", case=False, regex=True
        )
        df = df[df["time_h"].notna() & ~controls].copy()
        for c in OBS_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna(subset=OBS_COLS)
        df["total"] = df[OBS_COLS].sum(axis=1)
        df = df[df["total"] > 0].copy()
        df["intact"] = df["Intact"] / df["total"]
        df["dsb"] = (df["Direct DSB"] + df["Processed DSB"]) / df["total"]
        df["indels"] = df["Indels"] / df["total"]
        df["target"] = target
        rows.append(df[["target", "sample", "time_h", "intact", "dsb", "indels"]])
    return pd.concat(rows, ignore_index=True)


def rhs(_t, y, p):
    S, E, Id, Ip, R, Ins, Del, MH = y
    kon, koff, kcut, kproc, kprec, kins, kdel, kmh, krebind = (
        p[k] for k in ["k_on", "k_off", "k_cut", "k_proc", "k_precise",
                       "k_ins", "k_del", "k_mh", "k_rebind"]
    )
    dS = -kon*S + koff*E + krebind*R
    dE = kon*S - (koff+kcut)*E
    dId = kcut*E - (kproc+kprec)*Id
    dIp = kproc*Id - (kprec+kins+kdel+kmh)*Ip
    dR = kprec*(Id+Ip) - krebind*R
    return [dS, dE, dId, dIp, dR, kins*Ip, kdel*Ip, kmh*Ip]


def simulate(times, p):
    times = np.asarray(times, dtype=float)
    unique = np.unique(np.concatenate(([0.0], times)))
    sol = solve_ivp(
        lambda t, y: rhs(t, y, p), (0.0, float(unique[-1])),
        [1., 0., 0., 0., 0., 0., 0., 0.], t_eval=unique,
        rtol=1e-8, atol=1e-10
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    lookup = {float(t): sol.y[:, i] for i, t in enumerate(sol.t)}
    return np.vstack([lookup[float(t)] for t in times])


def aggregate(states):
    # Approximate mapping to the broad classes measured in the source workbook.
    a = np.column_stack([
        states[:, 0] + states[:, 1] + states[:, 4],
        states[:, 2] + states[:, 3],
        states[:, 5] + states[:, 6] + states[:, 7]
    ])
    return a / np.maximum(a.sum(axis=1, keepdims=True), 1e-12)


def initial_params(target):
    table = {
        "Psy1": dict(k_on=.08, k_off=.12, k_cut=.20, k_proc=.09,
                     k_precise=.05, k_ins=.02, k_del=.04, k_mh=.01, k_rebind=.01),
        "CRTISO": dict(k_on=.10, k_off=.10, k_cut=.25, k_proc=.12,
                       k_precise=.04, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01),
        "PhyB2": dict(k_on=.15, k_off=.08, k_cut=.35, k_proc=.15,
                      k_precise=.03, k_ins=.02, k_del=.05, k_mh=.01, k_rebind=.01)
    }
    return table[target].copy()


def fit_macro_rates(target, observed):
    p = initial_params(target)
    times = observed["time_h"].to_numpy(float)
    yobs = observed[["intact", "dsb", "indels"]].to_numpy(float)
    keys = ["k_on", "k_cut", "k_proc"]

    def residual(theta):
        pp = p.copy()
        for key, value in zip(keys, np.exp(theta)):
            pp[key] = float(value)
        return (aggregate(simulate(times, pp)) - yobs).ravel()

    result = least_squares(
        residual, np.log([p[k] for k in keys]),
        bounds=(np.log([1e-5]*3), np.log([2.0]*3)), max_nfev=4000
    )
    for key, value in zip(keys, np.exp(result.x)):
        p[key] = float(value)
    return p, bool(result.success)


def main():
    raw = load_data()
    raw.to_csv(OUTDIR / "experimental_data_used.csv", index=False)
    summary = raw.groupby(["target", "time_h"], as_index=False).agg(
        intact=("intact", "mean"), dsb=("dsb", "mean"),
        indels=("indels", "mean"), n_replicates=("sample", "count")
    )
    summary.to_csv(OUTDIR / "observed_summary.csv", index=False)

    grid = np.linspace(0, 72, 361)
    colors = ["#1971c2", "#fd7e14", "#2b8a3e", "#e03131",
              "#845ef7", "#795548", "#f06595", "#868e96"]
    trajectories, params_rows = [], []

    for target in TARGETS:
        obs = summary[summary["target"] == target].sort_values("time_h")
        params, success = fit_macro_rates(target, obs)
        states = np.maximum(simulate(grid, params), 0)
        states_pct = states / np.maximum(states.sum(axis=1, keepdims=True), 1e-12) * 100

        # Save all states to CSV.
        for j, (short, long_name) in enumerate(STATES):
            for t, value in zip(grid, states_pct[:, j]):
                trajectories.append({
                    "target": target, "time_h": t, "state": short,
                    "state_description": long_name, "model_percent": value
                })

            # One separate plot per compartment per target.
            fig, ax = plt.subplots(figsize=(8, 4.8))
            ax.plot(grid, states_pct[:, j], color=colors[j], lw=2.5)
            ax.set_title(f"{target}: {long_name}")
            ax.set_xlabel("Time (hours)")
            ax.set_ylabel("% of model pool (predicted)")
            ax.set_xlim(0, 72)
            ax.grid(alpha=.25)
            ax.text(
                .02, .96,
                "Model prediction — this compartment is not directly measured separately",
                transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=.3", facecolor="white", alpha=.8)
            )
            fig.tight_layout()
            fig.savefig(OUTDIR / f"{target}_{short}_separate.png", dpi=200, bbox_inches="tight")
            plt.close(fig)

        # One 8-panel plot per target for easy understanding.
        fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
        for j, ax in enumerate(axes.flat):
            ax.plot(grid, states_pct[:, j], color=colors[j], lw=2)
            ax.set_title(STATES[j][1], fontsize=9)
            ax.set_xlabel("Time (h)")
            ax.set_ylabel("% predicted")
            ax.grid(alpha=.2)
        fig.suptitle(f"{target}: eight compartments shown separately", y=1.02)
        fig.tight_layout()
        fig.savefig(OUTDIR / f"{target}_all_8_compartments_panel.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        params_rows.append({
            "target": target, **params, "macro_fit_success": success,
            "note": "Only k_on, k_cut, k_proc fitted; other rates are assumptions."
        })

    pd.DataFrame(trajectories).to_csv(OUTDIR / "all_8_state_trajectories.csv", index=False)
    pd.DataFrame(params_rows).to_csv(OUTDIR / "parameters_fitted_and_assumed.csv", index=False)
    (OUTDIR / "README.txt").write_text(
        "Separate eight-compartment plots for learning/interpretation.\\n"
        "Individual curves are model predictions, not direct experimental measurements.\\n"
        "Only k_on, k_cut, k_proc are fitted to aggregate observed classes; other rates are assumptions.\\n",
        encoding="utf-8"
    )
    print("DONE! All separate plots and CSV results saved to:")
    print(OUTDIR)
    print("Created 24 individual compartment plots + 3 eight-panel plots.")
    print("See README.txt for the scientific limitations.")


if __name__ == "__main__":
    main()


DONE! All separate plots and CSV results saved to:
C:\Users\lenovo\Desktop\sb miniproject\final_results\separate_compartment_plots
Created 24 individual compartment plots + 3 eight-panel plots.
See README.txt for the scientific limitations.
